# ⚡ KINETIC GPU Provider Node

Run this notebook on Google Colab (with GPU runtime) to become a **free GPU provider** on the KINETIC decentralized compute network.

**Prerequisites:**
- Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU
- Have an Algorand TestNet wallet funded via [faucet](https://bank.testnet.algorand.network/)
- Your KINETIC hub URL (local ngrok or deployed)

In [ ]:
# Cell 1: Install dependencies
!pip install flask pyngrok requests torch -q
print("✅ Dependencies installed")

In [ ]:
# Cell 2: Configuration — EDIT THESE VALUES
import os

# === REQUIRED: Set your values ===
os.environ["PROVIDER_WALLET"] = ""        # Your Algorand TestNet address
os.environ["PROVIDER_MNEMONIC"] = ""       # 25-word mnemonic for on-chain registration
os.environ["HUB_URL"] = "http://localhost:8000"  # Your KINETIC hub URL

# === OPTIONAL: Customize ===
os.environ["NODE_ID"] = "colab-gpu-t4"
os.environ["GPU_MODEL"] = "T4"
os.environ["VRAM_GB"] = "16"
os.environ["PORT"] = "5001"
os.environ["PRICE_PER_HOUR"] = "0.5"      # ALGO per hour
os.environ["ORG_NAME"] = "Colab GPU Node"

print("✅ Configuration set")

In [ ]:
# Cell 3: Verify GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU Available: {gpu_name} ({vram:.1f} GB)")
    print(f"   CUDA Version: {torch.version.cuda}")
    os.environ["GPU_MODEL"] = gpu_name.split()[-1]  # e.g. "T4"
    os.environ["VRAM_GB"] = str(int(vram))
else:
    print("⚠️  No GPU detected! Set runtime to GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 4: Create the provider server
%%writefile /content/provider_server.py
import hashlib, json, os, time, uuid
from flask import Flask, request, jsonify

app = Flask(__name__)
NODE_ID = os.getenv("NODE_ID", "colab-gpu")
GPU_MODEL = os.getenv("GPU_MODEL", "T4")
VRAM_GB = int(os.getenv("VRAM_GB", "16"))
PROVIDER_WALLET = os.getenv("PROVIDER_WALLET", "")
jobs = {}
stats = {"start": time.time(), "completed": 0, "total_ms": 0}

@app.route("/health")
def health():
    import torch
    return jsonify({
        "status": "active", "node_id": NODE_ID,
        "gpu_model": torch.cuda.get_device_name(0) if torch.cuda.is_available() else GPU_MODEL,
        "vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1) if torch.cuda.is_available() else VRAM_GB,
        "gpu_available": torch.cuda.is_available(),
        "uptime_seconds": int(time.time() - stats["start"]),
        "jobs_completed": stats["completed"],
    })

@app.route("/capabilities")
def capabilities():
    import torch
    return jsonify({
        "node_id": NODE_ID, "gpu_model": GPU_MODEL,
        "vram_gb": VRAM_GB, "gpu_available": torch.cuda.is_available(),
        "supported_tasks": ["sha256_compute", "inference", "gpu_matmul", "python_exec"],
        "wallet": PROVIDER_WALLET,
    })

@app.route("/providers/me")
def providers_me():
    return jsonify({
        "vram_gb": VRAM_GB, "gpu_model": GPU_MODEL,
        "price_per_hour": 100, "uptime_score": 100,
        "payment_address": PROVIDER_WALLET, "wallet": PROVIDER_WALLET,
        "node_id": NODE_ID,
    })

@app.route("/job", methods=["POST"])
def run_job():
    data = request.json or {}
    job_id = data.get("job_id") or str(uuid.uuid4())
    payload = str(data.get("payload", ""))
    tokens = int(data.get("tokens", 0))
    task_type = data.get("type", "inference")
    start = time.time()
    
    if task_type == "gpu_matmul":
        import torch
        size = max(64, min(tokens, 4096))
        a = torch.randn(size, size, device="cuda")
        b = torch.randn(size, size, device="cuda")
        torch.cuda.synchronize()
        c = torch.matmul(a, b)
        torch.cuda.synchronize()
        output = f"matmul_{size}x{size}_sum={c.sum().item():.6f}"
        method = "gpu_cuda"
    elif task_type in ("inference", "gpu_inference"):
        try:
            import torch
            seq_len = max(1, min(tokens, 2048))
            x = torch.randn(1, seq_len, 768, device="cuda")
            w = torch.randn(768, 768, device="cuda")
            torch.cuda.synchronize()
            out = torch.matmul(x, w).mean().item()
            torch.cuda.synchronize()
            output = f"inference_result={out:.8f}_tokens={seq_len}"
            method = "gpu_inference"
        except Exception:
            output = payload
            for _ in range(min(tokens, 1000)):
                output = hashlib.sha256(output.encode()).hexdigest()
            method = "cpu_sha256"
    else:
        output = payload
        for _ in range(min(tokens, 1000)):
            output = hashlib.sha256(output.encode()).hexdigest()
        method = "cpu_sha256"
    
    duration_ms = int((time.time() - start) * 1000)
    result_hash = hashlib.sha256(output.encode()).hexdigest()
    stats["completed"] += 1
    stats["total_ms"] += duration_ms
    
    return jsonify({
        "job_id": job_id, "result_hash": result_hash,
        "output": output[:200], "compute_output": output,
        "tokens_processed": tokens, "duration_ms": duration_ms,
        "execution_method": method, "node_id": NODE_ID,
    })

@app.route("/job/<job_id>/status")
def job_status(job_id):
    return jsonify({"job_id": job_id, "status": "completed"})

print("✅ Provider server code written")

In [ ]:
# Cell 5: Start ngrok tunnel + Flask server
from pyngrok import ngrok
import threading

# Set your ngrok auth token (get free at https://dashboard.ngrok.com/)
# Uncomment and set if needed:
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

# Start ngrok tunnel
public_url = ngrok.connect(5001)
print(f"\n{'='*60}")
print(f"  ⚡ KINETIC Provider Node")
print(f"{'='*60}")
print(f"  Public URL : {public_url}")
print(f"  Node ID    : {os.environ.get('NODE_ID', 'colab-gpu')}")
print(f"  GPU        : {os.environ.get('GPU_MODEL', 'T4')}")
print(f"{'='*60}")
print(f"  📋 Copy this URL into your KINETIC hub as PROVIDER_ENDPOINT")
print(f"{'='*60}\n")

# Save for registration
os.environ["PROVIDER_ENDPOINT"] = str(public_url)

# Start Flask in background thread
def run_server():
    import importlib.util
    spec = importlib.util.spec_from_file_location("provider_server", "/content/provider_server.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.app.run(host="0.0.0.0", port=5001, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time; time.sleep(3)  # Wait for server to start

# Test local endpoint
import requests
try:
    r = requests.get("http://localhost:5001/health", timeout=5)
    print(f"✅ Server running: {r.json()['status']}")
except Exception as e:
    print(f"⚠️  Server test failed: {e}")

In [ ]:
# Cell 6: Register with KINETIC Hub (on-chain)
import requests

hub_url = os.environ.get("HUB_URL", "http://localhost:8000")
provider_endpoint = os.environ.get("PROVIDER_ENDPOINT", "")
provider_mnemonic = os.environ.get("PROVIDER_MNEMONIC", "")

if not provider_mnemonic:
    print("⚠️  Set PROVIDER_MNEMONIC in Cell 2 for on-chain registration")
    print(f"   Your provider is still accessible at: {provider_endpoint}")
else:
    try:
        resp = requests.post(f"{hub_url}/provider/register", json={
            "gpu_model": os.environ.get("GPU_MODEL", "T4"),
            "vram_gb": int(os.environ.get("VRAM_GB", "16")),
            "price_per_hour": float(os.environ.get("PRICE_PER_HOUR", "0.5")),
            "endpoint": provider_endpoint,
            "provider_mnemonic": provider_mnemonic,
            "org_name": os.environ.get("ORG_NAME", "Colab GPU Node"),
        }, timeout=30)
        
        if resp.status_code == 200:
            result = resp.json()
            print(f"✅ Registered on-chain!")
            print(f"   TX: {result.get('explorer_url', 'N/A')}")
            print(f"   Provider: {result.get('provider_address', 'N/A')}")
        else:
            print(f"⚠️  Registration response: {resp.status_code} — {resp.text[:200]}")
    except Exception as e:
        print(f"⚠️  Could not reach hub at {hub_url}: {e}")
        print(f"   Provider is still accessible at: {provider_endpoint}")

# Also send heartbeat to hub
try:
    resp = requests.post(f"{hub_url}/providers/heartbeat", json={
        "endpoint": provider_endpoint,
        "vram_gb": int(os.environ.get("VRAM_GB", "16")),
        "gpu_model": os.environ.get("GPU_MODEL", "T4"),
        "gpu_available": True,
        "node_id": os.environ.get("NODE_ID", "colab-gpu"),
    }, timeout=10)
    if resp.status_code == 200:
        print(f"✅ Heartbeat sent to hub")
except Exception:
    pass

In [ ]:
# Cell 7: Keep alive + periodic heartbeat
import time, requests, threading

def heartbeat_loop():
    hub_url = os.environ.get("HUB_URL", "http://localhost:8000")
    endpoint = os.environ.get("PROVIDER_ENDPOINT", "")
    while True:
        try:
            requests.post(f"{hub_url}/providers/heartbeat", json={
                "endpoint": endpoint,
                "vram_gb": int(os.environ.get("VRAM_GB", "16")),
                "gpu_model": os.environ.get("GPU_MODEL", "T4"),
                "gpu_available": True,
                "node_id": os.environ.get("NODE_ID", "colab-gpu"),
            }, timeout=10)
        except Exception:
            pass
        time.sleep(30)

hb_thread = threading.Thread(target=heartbeat_loop, daemon=True)
hb_thread.start()

print("🟢 Provider running! Heartbeat active.")
print(f"   URL: {os.environ.get('PROVIDER_ENDPOINT', 'N/A')}")
print("\n   Keep this cell running. The provider will stay active")
print("   as long as this Colab session is alive.")
print("\n   Press Ctrl+C or stop the cell to shut down.")

# Keep alive
try:
    while True:
        time.sleep(60)
        r = requests.get("http://localhost:5001/health", timeout=5)
        data = r.json()
        print(f"   ♥ Uptime: {data.get('uptime_seconds', 0)//60}m | Jobs: {data.get('jobs_completed', 0)}")
except KeyboardInterrupt:
    print("\n🛑 Provider shut down")